In [2]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
from datetime import datetime, timedelta
import os
import requests
import pandas as pd
import time


In [3]:
def get_token(secret: str, url: str = "https://dep2.simondg.com/auth/login"):
    """Function for POST request to get access token using secret"""
    response = requests.post(url, json={"secret": secret})
    response.raise_for_status()  # Raises an error if the request fails
    return response.json()["access_token"]


def get_subgroup(token: str, subgroup_id: int):
    """Fetch students for a given subgroup"""
    url = f"https://dep2.simondg.com/students/{subgroup_id}"
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return pd.DataFrame(response.json()["students"])

In [20]:
secret = os.getenv("SECRET")
if not secret:
    raise ValueError("SECRET not found in .env file")

token = get_token(secret)

# Example subgroup IDs
subgroup_ids = [6082490,5995907]


# subgroups dataframes
dfs = [get_subgroup(token, sg_id) for sg_id in subgroup_ids]

In [21]:
dfs

[   SUBGROEPID SUBGROEPCODE  DEELGROEPID  \
 0     6082490    MC/DATA/1        27495   
 1     6082490    MC/DATA/1        27495   
 2     6082490    MC/DATA/1        27495   
 3     6082490    MC/DATA/1        27495   
 
                                                 NAAM  \
 0  u_40d1c0b4832f87c5caf2b9226e79b454219c70f90698...   
 1  u_bd60de66bbede5b67ec4757d5d1582bf849ba88a6598...   
 2  u_5573097a6230aedef054da9d0faa056d32dddef571e7...   
 3  u_daaa136a11fdb4e8e6abcfa62a8ac6308c54b1f63bc8...   
 
                                                EMAIL  
 0  u_e380c00a22b1d8c1ad26a11849fcc46151d31b5c8223...  
 1  u_96c45275c04234c761ef2980da9a0a783f11e8b6cef1...  
 2  u_107e217017cfa95d7247a2df030b70bad83fb6d3b703...  
 3  u_edcb94b2ef3b52209cb2d51562c37089a6b444684af6...  ,
     SUBGROEPID    SUBGROEPCODE  DEELGROEPID  \
 0      5995907  PBA-TIN-TI/3C2        26689   
 1      5995907  PBA-TIN-TI/3C2        26689   
 2      5995907  PBA-TIN-TI/3C2        26689   
 3      5995907  P

In [22]:
df_all = pd.concat(dfs, ignore_index=True)
df_all

,SUBGROEPID,SUBGROEPCODE,DEELGROEPID,NAAM,EMAIL
0,6082490,MC/DATA/1,27495,u_40d1c0b4832f87c5caf2b9226e79b454219c70f90698...,u_e380c00a22b1d8c1ad26a11849fcc46151d31b5c8223...
1,6082490,MC/DATA/1,27495,u_bd60de66bbede5b67ec4757d5d1582bf849ba88a6598...,u_96c45275c04234c761ef2980da9a0a783f11e8b6cef1...
2,6082490,MC/DATA/1,27495,u_5573097a6230aedef054da9d0faa056d32dddef571e7...,u_107e217017cfa95d7247a2df030b70bad83fb6d3b703...
3,6082490,MC/DATA/1,27495,u_daaa136a11fdb4e8e6abcfa62a8ac6308c54b1f63bc8...,u_edcb94b2ef3b52209cb2d51562c37089a6b444684af6...
4,5995907,PBA-TIN-TI/3C2,26689,u_fa8413bab74129cffc039f532092cecd4d412f98babf...,u_d2c4f6edaa136ec1b7d3202c2bc9716e525f5e55fcab...
5,5995907,PBA-TIN-TI/3C2,26689,u_2434d5ed68d17125281a772d3281245b9d4864fb698b...,u_bd00b117a705943034d30d2b54e92879b5c4f0e3ce37...
6,5995907,PBA-TIN-TI/3C2,26689,u_a8faf869ddf41f8520f0679ef53fbdb80a3be4faa9f4...,u_352a9a0efbc6cb77cfd5cdcc7e5c2674dc15ff5327b1...
7,5995907,PBA-TIN-TI/3C2,26689,u_1857708023cb428a0d610bd9be4a22762a3dd450a8dc...,u_8c36f5fb2310cdc9bbc33fb6e883f77180d2153323bc...
8,5995907,PBA-TIN-TI/3C2,26689,u_cc8e46d20e27b86c736d0cee7f7706dad8668dbc2c0f...,u_2f737d92e09cf0e82755c6151c816ef08e2f42811b74...
9,5995907,PBA-TIN-TI/3C2,26689,u_e2d81ed2fdf955c46243cd2e5b12225590cad199e98a...,u_9e39729430e1932d192d124846e006b3be85a21c580f...


In [23]:
# Drop duplicates by a unique identifier (EMAIL or NAAM)
df_unique = df_all.drop_duplicates(subset=["EMAIL"])

# Show total unique students
print(f"Total unique students across all subgroups: {len(df_unique)}")

# Optional: show counts per subgroup and overall
print(df_all.groupby("SUBGROEPID").size())
df_unique

Total unique students across all subgroups: 41
SUBGROEPID
5995907    37
6082490     4
dtype: int64


,SUBGROEPID,SUBGROEPCODE,DEELGROEPID,NAAM,EMAIL
0,6082490,MC/DATA/1,27495,u_40d1c0b4832f87c5caf2b9226e79b454219c70f90698...,u_e380c00a22b1d8c1ad26a11849fcc46151d31b5c8223...
1,6082490,MC/DATA/1,27495,u_bd60de66bbede5b67ec4757d5d1582bf849ba88a6598...,u_96c45275c04234c761ef2980da9a0a783f11e8b6cef1...
2,6082490,MC/DATA/1,27495,u_5573097a6230aedef054da9d0faa056d32dddef571e7...,u_107e217017cfa95d7247a2df030b70bad83fb6d3b703...
3,6082490,MC/DATA/1,27495,u_daaa136a11fdb4e8e6abcfa62a8ac6308c54b1f63bc8...,u_edcb94b2ef3b52209cb2d51562c37089a6b444684af6...
4,5995907,PBA-TIN-TI/3C2,26689,u_fa8413bab74129cffc039f532092cecd4d412f98babf...,u_d2c4f6edaa136ec1b7d3202c2bc9716e525f5e55fcab...
5,5995907,PBA-TIN-TI/3C2,26689,u_2434d5ed68d17125281a772d3281245b9d4864fb698b...,u_bd00b117a705943034d30d2b54e92879b5c4f0e3ce37...
6,5995907,PBA-TIN-TI/3C2,26689,u_a8faf869ddf41f8520f0679ef53fbdb80a3be4faa9f4...,u_352a9a0efbc6cb77cfd5cdcc7e5c2674dc15ff5327b1...
7,5995907,PBA-TIN-TI/3C2,26689,u_1857708023cb428a0d610bd9be4a22762a3dd450a8dc...,u_8c36f5fb2310cdc9bbc33fb6e883f77180d2153323bc...
8,5995907,PBA-TIN-TI/3C2,26689,u_cc8e46d20e27b86c736d0cee7f7706dad8668dbc2c0f...,u_2f737d92e09cf0e82755c6151c816ef08e2f42811b74...
9,5995907,PBA-TIN-TI/3C2,26689,u_e2d81ed2fdf955c46243cd2e5b12225590cad199e98a...,u_9e39729430e1932d192d124846e006b3be85a21c580f...
